# Analysis of the effect of interconference play on ranking methods
John S. McAlister - *Princeton University*<br>
Jack Haight - *University of Tennessee - Knoxville*<br>
Mate Szurop - *University of Tennessee - Chattenooga*<br>
<br><br>
SEC fans and B1G fans both believe that their conference is home to the best teams in college football. Here we investigate the ways that interconference play (or the lack thereof) can change how teams are ranked. The function demonstrated here are found in PairwiseComparision.py, in the same repository. 

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import PairwiseComparison as pc

## How many interconference games is enough?
The main question at hand is how many interconference games is sufficient to be sure that our rankings are "good enough." In order to investigate this, we will generate schedules featuring two conferences with some number of conference games, $g$, and some number of interconference games, $M$. We will examine how changing $M$ can change the quality of our ranking. The first challenge is to generate a schedule and simulate play <br>
In the method simulate.py, a schedule is generated according with two conferences one with $n$ teams, the other with $m$ teams each playing $g$ conference games. Then $M$ total non-conference games are added randomly between members of opposite conferences. This schedule looks like an adjacency matrix (because it is).<br> The simulation also randomly generates team "strengths" (see Newman 2023 for more details) and based on these strengths, scores for each game are computed. In the below example conference 1 has 10 teams, conference 2 has 12 teams and there will be 

In [ ]:
result = pc.simulate(10,12,10,dist = 'exp')

The score matrix looks like this

In [ ]:
W=result[0]
print(W)

and the strengths look like this

In [ ]:
tp = result[1]
print(tp)

Having the strengths like this means that we can give each team a rank which is the "base truth" we will see how well our ranking algorithm can recover this base truth. Below is listed the rank of each team (teams 1-10 are in conference 1 and teams 11-22 are in conference 2)

In [ ]:
truerank = np.argsort(np.argsort(tp))
print(truerank)

Now we can run the rank algorithm to try to see if we can recover this base truth from the randomly generated score matrix. 

In [ ]:
ranking = pc.Rank(W)
foundrank = np.argsort(ranking)
print(foundrank)

Now we want to compare the two rankings. There are many ways that we can do that. Right now, we have implemented L1 distance, but we may also be interested to know if the top team, top 4 teams, or top 12 teams were found correctly in the right order

In [ ]:
print(truerank)
print(foundrank)
print('L1 distance')
print(pc.L1dist(truerank,foundrank))
print('Did we find the top team?')
print(pc.correct_winner(truerank,foundrank))
print('Did we find the top 4 teams?')
print(pc.correct_top_four(truerank,foundrank))
print('Did we find the top 12 teams?')
print(pc.correct_top_12(truerank,foundrank))

## Changing the number of interconference games

Now lets see what how these distance metrics change as we very the number of interconference games

In [ ]:
results = pc.scan(10,12,0,20,reps = 400)
ms = results[0]
lds = results[1]
cws = results[2]
ct4 = results[3]
ct12 = results[4]

plt.plot(ms,lds)
plt.xlabel('Number of interconference Games')
plt.ylabel('L1 distance')
plt.title('L1 distance between found and true rankings')
plt.show()

In [ ]:
fig = plt.figure
ax = plt.subplot(111)
line1 = ax.plot(ms,cws, label = 'Single Champion')
line2 = ax.plot(ms,ct4, label = '4 team playoff')
line3 = ax.plot(ms,ct12, label = '12 team playoff')
ax.set(xlabel='Number of interconference games', ylabel ='probability', title = 'Probability of correctly seeding a playoff')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15),
          fancybox=True, shadow=True, ncol=3)